# Retrieval Experimentation Notebook

This notebook evaluates different retrieval approaches for the Yoga RAG system:

- Text-based search (BM25/TF-IDF)
- Vector-based semantic search
- Hybrid search (combination of text and vector)

We'll use the ground truth dataset to calculate:

- **Hit Rate**: Percentage of queries where at least one relevant document is in top-k results
- **MRR (Mean Reciprocal Rank)**: Average of 1/rank of first relevant document

Target metrics: Hit Rate > 90%, MRR > 0.85


## Setup and Imports


In [ ]:
import pandas as pd
import numpy as np
from typing import List, Dict, Tuple
import warnings

warnings.filterwarnings("ignore")

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

## Load Data


In [ ]:
# Load yoga poses dataset
yoga_data = pd.read_csv("../data/yoga_data_merged.csv")
print(f"Loaded {len(yoga_data)} yoga poses")
print(f"\nColumns: {list(yoga_data.columns)}")
print(f"\nFirst few rows:")
yoga_data.head()

Loaded 202 yoga poses

Columns: ['id', 'pose_name', 'sanskrit_name', 'category', 'difficulty_level', 'benefits', 'contraindications', 'breathing_pattern', 'duration_or_reps', 'modifications', 'instructions']

First few rows:


,id,pose_name,sanskrit_name,category,difficulty_level,benefits,contraindications,breathing_pattern,duration_or_reps,modifications,instructions
0,1,Mountain Pose,Tadasana,standing,beginner,"Mountain Pose strengthens the ankles, calves, and spine, improving overall posture and balance. ...","Avoid practicing Mountain Pose if you have severe ankle or knee injuries, or if you experience d...",Inhale: Feel the chest expand and the spine lengthen. Exhale: Engage the core and maintain good ...,"Hold Mountain Pose for 3-5 breaths, or as long as you feel comfortable and focused. Repeat as ne...","For beginners, use a block or a wall for support and balance. If you have any knee or ankle issu...","1. Stand with your feet hip-width apart, parallel to each other, and facing forward. 2. Engage y..."
1,2,Tree Pose,Vrksasana,balancing,beginner,"Tree Pose (Vrksasana) improves balance, stability, and focus. It strengthens the ankles and calv...","Avoid Tree Pose if you have ankle or knee injuries, or if you experience dizziness or lightheade...","Inhale deeply and lengthen your spine, exhale slowly and engage your core. Continue to breathe n...","Hold Tree Pose for 30 seconds to 1 minute on each leg, or for 3-5 breaths if you're just startin...",Use a block or wall for support if you're new to balancing poses. You can also practice Tree Pos...,"1. Start by standing on one leg, with the other foot resting against your inner thigh. Engage yo..."
2,3,Downward-Facing Dog,Adho Mukha Svanasana,standing,beginner,"Downward-Facing Dog stretches and strengthens the entire back side of the body, from the shoulde...","Avoid this pose if you have recent injuries to the shoulders, wrists, or ankles, or if you have ...","Inhale as you lift your hips up and back, exhale as you lengthen your spine and engage your core.","Hold for 3-5 breaths, or for 30-60 seconds. Repeat 2-3 times.","For beginners or those with wrist issues, try using blocks or a strap to support the hands. For ...","1. Start on all fours, with your hands shoulder-width apart and your knees directly under your h..."
3,4,Cobra Pose,Bhujangasana,backbend,beginner,"Cobra Pose strengthens the back muscles, opens the chest, and improves flexibility in the should...","Avoid Cobra Pose if you have a recent back injury, herniated disk, or any serious spinal conditi...","Inhale as you press your palms into the ground and lift your chest and head off the mat, keeping...","Hold Cobra Pose for 3-5 breaths, or 15-30 seconds, and repeat for 2-3 repetitions.","For a gentler variation, place your forearms on the ground instead of your palms, or use blocks ...",1. Lie on your stomach with your hands under your shoulders and your fingers spread wide. Engage...
4,5,Cat-Cow Pose,Marjaryasana-Bitilasana,standing,beginner,"The Cat-Cow Pose stretches the spine, neck, and torso, while also improving flexibility and redu...","This pose is generally safe for most people, but those with severe neck injuries or cervical spi...","Inhale as you arch your back and lift your head and tailbone (Cow Pose), exhale as you round you...","Repeat the sequence for 5-10 breaths, moving slowly and smoothly between the Cat and Cow poses.","For a more gentle version, try using a block or strap to support your neck and spine. You can al...","1. Start on your hands and knees, with your wrists directly under your shoulders and your knees ..."


In [ ]:
# Load ground truth evaluation dataset
ground_truth = pd.read_csv("../data/ground_truth.csv")
print(f"Loaded {len(ground_truth)} ground truth questions")
print(f"\nColumns: {list(ground_truth.columns)}")
print(f"\nFirst few rows:")
ground_truth.head()

Loaded 75 ground truth questions

Columns: ['question', 'expected_answer', 'relevant_pose_ids']

First few rows:


,question,expected_answer,relevant_pose_ids
0,What pose is great for improving balance and focus?,Tree Pose and Warrior Pose I are both great for improving balance and focus.,"2,6"
1,What are the benefits of Cobra Pose?,"Cobra Pose strengthens the back muscles, opens the chest, and improves flexibility in the should...",4
2,Can you give me some standing poses that are good for beginners?,"Mountain Pose, Downward-Facing Dog, and Warrior Pose I are all standing poses that are suitable ...","1,3,6"
3,How do I do a Cat-Cow Pose?,"Start on your hands and knees, then as you inhale, arch your back and lift your head and tailbon...",5
4,What poses should I avoid if I have a recent back injury?,"Avoid Cobra Pose, Seated Spinal Twist, and Bridge Pose if you have a recent back injury. Also, b...","4,8,9,7"


In [ ]:
# Parse relevant_pose_ids from string to list of integers
def parse_pose_ids(pose_ids_str: str) -> List[int]:
    """Convert comma-separated string of pose IDs to list of integers."""
    if pd.isna(pose_ids_str):
        return []
    return [int(x.strip()) for x in str(pose_ids_str).split(",")]


ground_truth["relevant_pose_ids_list"] = ground_truth[
    "relevant_pose_ids"
].apply(parse_pose_ids)
print(f"\nParsed relevant pose IDs:")
ground_truth[["question", "relevant_pose_ids_list"]].head()


Parsed relevant pose IDs:


,question,relevant_pose_ids_list
0,What pose is great for improving balance and focus?,"[2, 6]"
1,What are the benefits of Cobra Pose?,[4]
2,Can you give me some standing poses that are good for beginners?,"[1, 3, 6]"
3,How do I do a Cat-Cow Pose?,[5]
4,What poses should I avoid if I have a recent back injury?,"[4, 8, 9, 7]"


## Evaluation Metrics

We'll implement two key metrics for retrieval evaluation:

1. **Hit Rate**: Measures if at least one relevant document appears in the top-k results
2. **MRR (Mean Reciprocal Rank)**: Measures how high the first relevant document ranks


In [ ]:
def calculate_hit_rate(
    retrieved_ids: List[List[int]], relevant_ids: List[List[int]]
) -> float:
    """
    Calculate hit rate: percentage of queries where at least one relevant document is retrieved.

    Args:
        retrieved_ids: List of lists, where each inner list contains retrieved document IDs for a query
        relevant_ids: List of lists, where each inner list contains relevant document IDs for a query

    Returns:
        Hit rate as a float between 0 and 1
    """
    hits = 0
    for retrieved, relevant in zip(retrieved_ids, relevant_ids):
        # Check if any retrieved ID is in the relevant set
        if any(doc_id in relevant for doc_id in retrieved):
            hits += 1

    return hits / len(retrieved_ids) if len(retrieved_ids) > 0 else 0.0

In [ ]:
def calculate_mrr(
    retrieved_ids: List[List[int]], relevant_ids: List[List[int]]
) -> float:
    """
    Calculate Mean Reciprocal Rank (MRR): average of 1/rank of first relevant document.

    Args:
        retrieved_ids: List of lists, where each inner list contains retrieved document IDs for a query
        relevant_ids: List of lists, where each inner list contains relevant document IDs for a query

    Returns:
        MRR as a float between 0 and 1
    """
    reciprocal_ranks = []

    for retrieved, relevant in zip(retrieved_ids, relevant_ids):
        # Find the rank (1-indexed) of the first relevant document
        for rank, doc_id in enumerate(retrieved, start=1):
            if doc_id in relevant:
                reciprocal_ranks.append(1.0 / rank)
                break
        else:
            # No relevant document found in retrieved results
            reciprocal_ranks.append(0.0)

    return np.mean(reciprocal_ranks) if len(reciprocal_ranks) > 0 else 0.0

In [7]:
def evaluate_retrieval(
    retrieved_ids: List[List[int]], relevant_ids: List[List[int]]
) -> Dict[str, float]:
    """
    Calculate both hit rate and MRR for a set of retrieval results.

    Args:
        retrieved_ids: List of lists, where each inner list contains retrieved document IDs for a query
        relevant_ids: List of lists, where each inner list contains relevant document IDs for a query

    Returns:
        Dictionary with 'hit_rate' and 'mrr' keys
    """
    return {
        "hit_rate": calculate_hit_rate(retrieved_ids, relevant_ids),
        "mrr": calculate_mrr(retrieved_ids, relevant_ids),
    }

## Data Structures for Testing

Set up basic data structures that will be used across all retrieval experiments.


In [12]:
# Create a dictionary for quick pose lookup by ID
pose_dict = {row["id"]: row.to_dict() for _, row in yoga_data.iterrows()}
print(f"Created pose dictionary with {len(pose_dict)} poses")

# Example: Look up a pose
example_pose = pose_dict[1]
print(f"\nExample pose (ID=1):")
print(f"Name: {example_pose['pose_name']}")
print(f"Category: {example_pose['category']}")
print(f"Difficulty: {example_pose['difficulty_level']}")

Created pose dictionary with 202 poses

Example pose (ID=1):
Name: Mountain Pose
Category: standing
Difficulty: beginner


In [13]:
# Create searchable text for each pose by combining relevant fields
def create_searchable_text(pose: Dict) -> str:
    """
    Combine multiple fields into a single searchable text string.
    This will be used for both text and vector search.
    """
    fields = [
        pose.get("pose_name", ""),
        pose.get("sanskrit_name", ""),
        pose.get("category", ""),
        pose.get("difficulty_level", ""),
        pose.get("benefits", ""),
        pose.get("contraindications", ""),
        pose.get("instructions", ""),
    ]
    return " ".join(str(f) for f in fields if pd.notna(f))


# Add searchable text to each pose
for pose_id, pose in pose_dict.items():
    pose["searchable_text"] = create_searchable_text(pose)

print("Added searchable text to all poses")
print(f"\nExample searchable text (first 200 chars):")
print(pose_dict[1]["searchable_text"][:200] + "...")

Added searchable text to all poses

Example searchable text (first 200 chars):
Mountain Pose Tadasana standing beginner Mountain Pose strengthens the ankles, calves, and spine, improving overall posture and balance. It also helps establish good alignment and promotes mental clar...


In [14]:
# Extract test queries and their relevant pose IDs
test_queries = ground_truth["question"].tolist()
test_relevant_ids = ground_truth["relevant_pose_ids_list"].tolist()

print(f"Prepared {len(test_queries)} test queries")
print(f"\nExample query:")
print(f"Q: {test_queries[0]}")
print(f"Relevant pose IDs: {test_relevant_ids[0]}")

Prepared 75 test queries

Example query:
Q: What pose is great for improving balance and focus?
Relevant pose IDs: [2, 6]
